# Frost Bites Supermarket: Interactive Tables Hands-on Lab

## Setup: Build the Frost Bites webstore data

Search and replace `_xx` with your UserID. ex: `dba_xx` --> `dba_iron_man`

This notebook builds everything with plain SQL. No files to upload, no Postgres, nothing to install. Run it top to bottom. It takes about 20 seconds.

> **Note:** the recommendations table built here has the same columns as the one the Transactions Lab 3 builds in the same schema, so either lab can fill it (whichever runs last wins).

In [ ]:
%%sql -r dataframe_2
use role accountadmin;

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

all_users = ['USER_ABOMINATION', 'USER_ADAM_WARLOCK', 'USER_AGENT_VENOM', 'USER_ALAN_SCOTT', 'USER_ALFRED_PENNYWORTH', 'USER_AMADEUS_CHO', 'USER_AMAZO', 'USER_AMBUSH_BUG', 'USER_AMERICA_CHAVEZ', 'USER_ANGEL', 'USER_ANGSTROM_LEVY', 'USER_ANIMAL_MAN', 'USER_ANNIHILUS', 'USER_ANT_MAN', 'USER_ANTI_MONITOR', 'USER_APOCALYPSE', 'USER_AQUAMAN', 'USER_ARCADE', 'USER_ARES', 'USER_ARNIM_ZOLA', 'USER_ATLAS', 'USER_ATOM_SMASHER', 'USER_AVALANCHE', 'USER_AZRAEL', 'USER_BANE', 'USER_BARON_MORDO', 'USER_BARON_ZEMO', 'USER_BATGIRL', 'USER_BATMAN', 'USER_BATROC_THE_LEAPER', 'USER_BATWING', 'USER_BATWOMAN', 'USER_BEAST', 'USER_BEAST_BOY', 'USER_BETA_RAY_BILL', 'USER_BEYONDER', 'USER_BIG_BARDA', 'USER_BISHOP', 'USER_BLACK_ADAM', 'USER_BLACK_BOLT', 'USER_BLACK_CANARY', 'USER_BLACK_LIGHTNING', 'USER_BLACK_MANTA', 'USER_BLACK_PANTHER', 'USER_BLACK_WIDOW', 'USER_BLACKOUT', 'USER_BLADE', 'USER_BLOB', 'USER_BLOODWULF', 'USER_BLUE_BEETLE', 'USER_BLUE_MARVEL', 'USER_BOOSTER_GOLD', 'USER_BRAINIAC', 'USER_BRIT', 'USER_BROTHER_EYE', 'USER_BULLSEYE', 'USER_BURNOUT', 'USER_CABLE', 'USER_CANNONBALL', 'USER_CAPTAIN_AMERICA']
users = [x[5:] for x in all_users]

users[:5]

## 0.1 Create a warehouse for the lab

This is a normal (standard) warehouse. We use it to build tables.

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql('use database hol;').collect()
    session.sql(f'use schema schema_{user};').collect()
    session.sql(f"""
        create or replace warehouse std_wh_{user}
        with warehouse_size = 'xsmall'
        auto_suspend = 300
        auto_resume = true;
        """).collect()

## 0.2 Products: the exotic fruit aisle

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql(f'use hol.schema_{user}').collect()
    session.sql("""
        create or replace table products (
            product_id  int,
            name        varchar,
            category    varchar,
            origin      varchar,
            price       number(8,2),
            stock_qty   int,
            description varchar
        );
        """).collect()

    session.sql("""
                insert into products values
        (1,  'Dragon Fruit',  'tropical',  'Vietnam',     6.99, 120, 'Bright pink skin and white flesh with tiny black seeds. Mild and sweet, like kiwi meets pear.'),
        (2,  'Rambutan',      'tropical',  'Thailand',    8.49,  90, 'Hairy red shell with a juicy white fruit inside. Sweet and floral, a lot like lychee.'),
        (3,  'Mangosteen',    'tropical',  'Indonesia',  12.99,  40, 'Purple rind with snow-white segments. Sweet and tangy, called the queen of fruits.'),
        (4,  'Durian',        'tropical',  'Malaysia',   24.99,  15, 'Spiky, famously smelly and much loved. Rich, custard-like flesh.'),
        (5,  'Jackfruit',     'tropical',  'India',      18.99,  25, 'The largest tree fruit. Sweet yellow pods that taste like mango and banana.'),
        (6,  'Passion Fruit', 'tropical',  'Brazil',      1.99, 200, 'Wrinkly purple shell full of tart, fragrant pulp. Great in drinks and desserts.'),
        (7,  'Lychee',        'tropical',  'China',       7.99, 150, 'Thin red skin and juicy see-through flesh. Sweet with a hint of rose.'),
        (8,  'Longan',        'tropical',  'Thailand',    6.49, 110, 'Tan shell and clear, sweet flesh. The name means dragon eye.'),
        (9,  'Soursop',       'tropical',  'Mexico',     14.99,  30, 'Spiky green skin and creamy white flesh. Tastes like strawberry, pineapple and citrus.'),
        (10, 'Cherimoya',     'tropical',  'Peru',       11.99,  35, 'Green scaly skin, custard-soft inside. Often called the custard apple.'),
        (11, 'Sapodilla',     'tropical',  'Philippines', 5.99,  60, 'Brown skin and soft flesh that tastes like brown sugar and pear.'),
        (12, 'Yuzu',          'citrus',    'Japan',       4.99,  80, 'Small, bumpy and very fragrant. Tart juice used in sauces and drinks.'),
        (13, 'Buddha''s Hand','citrus',    'China',       9.99,  20, 'Finger-shaped citrus with no juice. Used for its strong lemon scent and zest.'),
        (14, 'Blood Orange',  'citrus',    'Italy',       1.49, 250, 'Deep red flesh with a sweet, berry-like taste.'),
        (15, 'Ugli Fruit',    'citrus',    'Jamaica',     2.99,  90, 'Wrinkly green-yellow skin. Sweet and juicy, like grapefruit crossed with mandarin.'),
        (16, 'Kumquat',       'citrus',    'China',       5.49, 100, 'Tiny oval citrus you eat whole. Sweet peel, sour center.'),
        (17, 'Calamansi',     'citrus',    'Philippines', 4.49, 120, 'Small green-orange lime. Tart, bright juice for drinks and cooking.'),
        (18, 'Finger Lime',   'citrus',    'Australia',  15.99,  25, 'Long thin lime full of tiny juice beads, known as citrus caviar.'),
        (19, 'Acai',          'berry',     'Brazil',      9.49,  70, 'Dark purple palm berry with an earthy taste. The base of the acai bowl.'),
        (20, 'Goji Berry',    'berry',     'China',       8.99,  85, 'Small red berry, a little sweet and a little tart. Usually eaten dried.'),
        (21, 'Goldenberry',   'berry',     'Colombia',    5.99,  95, 'Golden berry in a papery husk. Sweet and tangy, like a tropical tomato.'),
        (22, 'Mulberry',      'berry',     'Turkey',      6.99,  60, 'Soft dark berry, sweet and juicy, a bit like a blackberry.'),
        (23, 'Jabuticaba',    'berry',     'Brazil',     13.99,  20, 'Grows right on the tree trunk. Grape-like berry with sweet white flesh.'),
        (24, 'Kiwano',        'melon',     'New Zealand', 7.49,  45, 'Spiky orange horned melon with green jelly inside. Tastes like cucumber and banana.'),
        (25, 'Pepino Melon',  'melon',     'Chile',       4.99,  55, 'Small striped melon. Mild and juicy, between honeydew and cucumber.'),
        (26, 'Canary Melon',  'melon',     'Spain',       5.49,  40, 'Bright yellow skin and pale sweet flesh with a tangy finish.'),
        (27, 'Star Fruit',    'specialty', 'Malaysia',    3.49, 140, 'Slice it and you get stars. Crisp, juicy and lightly sweet-sour.'),
        (28, 'Persimmon',     'specialty', 'Japan',       2.49, 160, 'Orange and honey-sweet when ripe. Soft like jam or crisp like an apple.'),
        (29, 'Guava',         'specialty', 'Mexico',      2.99, 130, 'Pink or white flesh with a strong tropical smell. Sweet, with a pear-like texture.'),
        (30, 'Feijoa',        'specialty', 'New Zealand', 3.99,  75, 'Green egg-shaped fruit. Tastes like pineapple, guava and mint.'),
        (31, 'Tamarillo',     'specialty', 'Ecuador',     4.49,  50, 'Egg-shaped tree tomato. Tangy and savory-sweet, great in salsa.'),
        (32, 'Breadfruit',    'specialty', 'Jamaica',     6.99,  30, 'Starchy when green, sweet when ripe. Cooks up like a potato.'),
        (33, 'Prickly Pear',  'specialty', 'Mexico',      2.49, 110, 'Cactus fruit with magenta flesh. Sweet, like watermelon and bubblegum.'),
        (34, 'Atemoya',       'specialty', 'Australia',  10.99,  25, 'Cross of cherimoya and sugar apple. Creamy and sweet, a little like pina colada.'),
        (35, 'Snake Fruit',   'specialty', 'Indonesia',   7.99,  45, 'Scaly brown skin like a snake. Crunchy and sweet with a hint of pineapple.');
        """).collect()

In [ ]:
%%sql -r dataframe_1
show schemas in database hol;

## 0.3 Customers: 10,000 webstore shoppers

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql(f'use hol.schema_{user}').collect()
    session.sql("""
        create or replace table customers as
        select
            id                                                        as customer_id,
            'Shopper ' || id                                          as name,
            'shopper' || id || '@frostbites.example'                  as email,
            dateadd(day, -uniform(1, 730, random()), current_date())  as joined_on
        from (select row_number() over (order by seq4()) as id
              from table(generator(rowcount => 10000)));
        """).collect()

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql(f'use hol.schema_{user}').collect()
    # customer 1 is Maya, our favorite shopper
    session.sql("""
        update customers
        set name = 'Maya Patel', email = 'maya@frostbites.example'
        where customer_id = 1;
        """).collect()

## 0.4 Orders: 500,000 orders over the last 30 days

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql(f'use hol.schema_{user}').collect()
    session.sql("""
        create or replace table orders as
        select
            id                                       as order_id,
            uniform(1, 10000, random())              as customer_id,
            dateadd(second, -uniform(0, 30*24*60*60, random()), current_timestamp())::timestamp_ntz as ordered_at,
            'picked_up'                              as status
        from (select row_number() over (order by seq4()) as id
              from table(generator(rowcount => 500000)));
        """).collect()

## 0.5 Order items: about 2 million fruit lines

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql(f'use hol.schema_{user}').collect()
    session.sql("""
        create or replace table order_items as
        select
            i.order_id,
            i.product_id,
            sum(i.qty)           as qty,
            sum(i.qty) * p.price as line_total
        from (
            select
                uniform(1, 500000, random()) as order_id,
                uniform(1, 35, random())     as product_id,
                uniform(1, 4, random())      as qty
            from table(generator(rowcount => 2000000))
        ) i
        join products p on p.product_id = i.product_id
        group by i.order_id, i.product_id, p.price;
        """).collect()

## 0.6 Recommendations: "shoppers who buy X also buy Y"

A simple recommender built from what people buy together. Each shopper gets 5 suggestions based on their favorite fruit.

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql(f'use hol.schema_{user}').collect()
    session.sql("""
        create or replace table recommendations as
        with favorites as (
            select o.customer_id, oi.product_id
            from orders o
            join order_items oi on oi.order_id = o.order_id
            group by o.customer_id, oi.product_id
            qualify row_number() over (partition by o.customer_id order by sum(oi.qty) desc) = 1
        ),
        bought_together as (
            select a.product_id, b.product_id as also_bought_id, count(*) as times_together
            from order_items a
            join order_items b on a.order_id = b.order_id and a.product_id <> b.product_id
            group by a.product_id, b.product_id
        )
        select
            f.customer_id,
            bt.also_bought_id                                   as product_id,
            bt.times_together                                   as score,
            'Shoppers who buy ' || fav.name || ' also buy this' as reason,
            current_timestamp()::timestamp_ntz                  as generated_at
        from favorites f
        join bought_together bt on bt.product_id = f.product_id
        join products fav       on fav.product_id = f.product_id
        qualify row_number() over (partition by f.customer_id order by bt.times_together desc) <= 5;
        """).collect()

## 0.7 Verify

Expect: products 35, customers 10000, orders 500000, order_items ~1.9 million, recommendations ~50000

In [ ]:
for user in users:
    session.sql(f'use role dba_{user};').collect()
    session.sql(f'use hol.schema_{user}').collect()
    print(f'--- schema_{user} ---')
    print(session.sql("""
        select 'products' as tbl, count(*) as row_count from products
        union all select 'customers', count(*) from customers
        union all select 'orders', count(*) from orders
        union all select 'order_items', count(*) from order_items
        union all select 'recommendations', count(*) from recommendations;
        """).collect())

# 0.8 Streamlit app access
Grants for the streamlit app

In [ ]:
for user in users:
    session.sql(f'GRANT USAGE ON DATABASE INTERACTIVE_APP_DB TO ROLE DBA_{user};').collect()
    session.sql(f'GRANT USAGE ON SCHEMA INTERACTIVE_APP_DB.DEMO TO ROLE DBA_{user};').collect()
    session.sql(f'GRANT USAGE ON STREAMLIT INTERACTIVE_APP_DB.DEMO.INTERACTIVE_WAREHOUSE_DEMO TO ROLE DBA_{user};').collect()
    

**Setup complete.** Move on to `lab1.sql`.